# 06 Trend Analysis and Forecasting
## Nifty Financial Platform

This notebook analyzes historical trends and performs future forecasting for key financial metrics using time series models.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from dotenv import load_dotenv

# Settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

# Load environment variables
load_dotenv(dotenv_path='../.env')
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    print("ERROR: DATABASE_URL not found in .env file")
else:
    engine = create_engine(DATABASE_URL)
    print("Database engine created successfully.")

## 1. Fetch Time Series Data
We need historical revenue and profit data for all years.

In [ ]:
def fetch_trend_data(engine):
    query = """
    SELECT 
        dc.symbol, dc.company_name, dy.fiscal_year, 
        pl.revenue, pl.net_profit
    FROM fact_profit_loss pl
    JOIN dim_company dc ON pl.company_id = dc.company_id
    JOIN dim_year dy ON pl.year_id = dy.year_id
    ORDER BY dc.symbol, dy.fiscal_year
    """
    return pd.read_sql(query, engine)

df = fetch_trend_data(engine)
df.head()

## 2. Aggregate Market Trends
Overall revenue and profit growth of all companies in the warehouse.

In [ ]:
market_trend = df.groupby('fiscal_year')[['revenue', 'net_profit']].sum().reset_index()

fig, ax1 = plt.subplots()

ax1.set_xlabel('Fiscal Year')
ax1.set_ylabel('Total Revenue', color='tab:blue')
ax1.plot(market_trend['fiscal_year'], market_trend['revenue'], color='tab:blue', marker='o', label='Revenue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Total Net Profit', color='tab:green')
ax2.plot(market_trend['fiscal_year'], market_trend['net_profit'], color='tab:green', marker='s', label='Net Profit')
ax2.tick_params(axis='y', labelcolor='tab:green')

plt.title('Aggregate Market Revenue and Profit Trends')
fig.tight_layout()
plt.show()

## 3. Forecasting (Holt-Winters)
We will forecast the next 3 years of aggregate revenue.

In [ ]:
data = market_trend.set_index('fiscal_year')['revenue']

# Fit model
model = ExponentialSmoothing(data, trend='add', seasonal=None, damped_trend=True)
model_fit = model.fit()

# Forecast next 3 years
forecast_years = np.arange(data.index.max() + 1, data.index.max() + 4)
forecast = model_fit.forecast(3)

plt.plot(data.index, data.values, marker='o', label='Historical')
plt.plot(forecast_years, forecast.values, marker='x', linestyle='--', color='red', label='Forecast')
plt.title('Market Revenue Forecast (Next 3 Years)')
plt.xlabel('Fiscal Year')
plt.ylabel('Revenue')
plt.legend()
plt.show()

## 4. Company Specific Growth Trends

In [ ]:
def plot_company_trend(symbol, df):
    c_data = df[df['symbol'] == symbol]
    if c_data.empty: return
    
    plt.figure(figsize=(10, 5))
    plt.plot(c_data['fiscal_year'], c_data['net_profit'], marker='o', color='orange')
    plt.title(f"Net Profit Trend for {symbol}")
    plt.xlabel('Year')
    plt.ylabel('Net Profit')
    plt.show()

test_symbol = df['symbol'].iloc[0]
plot_company_trend(test_symbol, df)

## 5. Summary of Future Outlook
Calculated projected growth rates.

In [ ]:
proj_growth = (forecast.iloc[-1] - data.iloc[-1]) / data.iloc[-1] * 100
print(f"Projected Market Revenue Growth over the next 3 years: {proj_growth:.2f}%")